<a href="https://colab.research.google.com/github/MEDATTA0/math_ml-formative-3/blob/main/Part_2_Bayesian_sentiment_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")


DATA_PATH = "/content/drive/MyDrive/Math_ML_Formative_3/IMDB Dataset.csv"

import os
print("File found:", os.path.exists(DATA_PATH), "->", DATA_PATH)


Mounted at /content/drive
File found: True -> /content/drive/MyDrive/Math_ML_Formative_3/IMDB Dataset.csv


In [20]:
POSITIVE_KEYWORDS = ["excellent", "brilliant", "masterpiece", "wonderful"]
NEGATIVE_KEYWORDS = ["boring", "awful", "waste", "worst"]

Read and clean the reviews

'tokenize' turns one review into a clean set of words; 'load_review' does it for all 50,000.


In [21]:
import csv
import re

def tokenize(text):
    """Return the set of clean words in a review."""
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)
    return set(re.findall(r"[a-z']+", text))

def load_reviews(path):
    """Return a list of (words, sentiment) for every review."""
    reviews = []
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            reviews.append((tokenize(row["review"]), row["sentiment"].strip().lower()))
    return reviews

reviews = load_reviews(DATA_PATH)
print("Total reviews loaded:", len(reviews))

Total reviews loaded: 50000


Count
Totals for the whole corpus, and per-keyword counts.

- N = total reviews, P = positive reviews
- K = reviews containing the keyword, A = positive reviews containing it


In [22]:
def count_corpus(reviews):
    """Return N (total reviews) and P (positive reviews)."""
    total = len(reviews)
    total_pos = sum(1 for _, s in reviews if s == "positive")
    return total, total_pos

def count_keyword(reviews, keyword):
    """Return K (reviews with keyword) and A (positive reviews with keyword)."""
    kw = keyword.lower()
    with_kw = pos_with_kw = 0
    for words, sentiment in reviews:
        if kw in words:
            with_kw += 1
            if sentiment == "positive":
                pos_with_kw += 1
    return with_kw, pos_with_kw

N, P = count_corpus(reviews)
print(f"N = {N} reviews, P = {P} positive")
print(f"Prior P(Positive) = {P / N:.4f}")

N = 50000 reviews, P = 25000 positive
Prior P(Positive) = 0.5000


Bayes' Theorem

P(Positive | keyword) = P(keyword | Positive) × P(Positive) / P(keyword)


In [26]:
def bayes_positive_given_keyword(N, P, with_kw, pos_with_kw):
    prior      = P / N
    likelihood = pos_with_kw / P
    marginal   = with_kw / N
    posterior  = (likelihood * prior) / marginal
    return prior, likelihood, marginal, posterior

Results table

In [27]:
print(f"{'keyword':<13}{'P(Pos)':>9}{'P(kw|Pos)':>11}{'P(kw)':>9}{'P(Pos|kw)':>11}")
print("-" * 53)
for keyword in POSITIVE_KEYWORDS + NEGATIVE_KEYWORDS:
    with_kw, pos_with_kw = count_keyword(reviews, keyword)
    if with_kw == 0:
        print(f"{keyword:<13}{'(not found)':>40}")
        continue
    prior, like, marg, post = bayes_positive_given_keyword(N, P, with_kw, pos_with_kw)
    print(f"{keyword:<13}{prior:>9.4f}{like:>11.4f}{marg:>9.4f}{post:>11.4f}")



keyword         P(Pos)  P(kw|Pos)    P(kw)  P(Pos|kw)
-----------------------------------------------------
excellent       0.5000     0.1146   0.0710     0.8073
brilliant       0.5000     0.0634   0.0417     0.7606
masterpiece     0.5000     0.0350   0.0240     0.7300
wonderful       0.5000     0.0902   0.0555     0.8120
boring          0.5000     0.0236   0.0610     0.1938
awful           0.5000     0.0112   0.0575     0.0977
waste           0.5000     0.0070   0.0507     0.0687
worst           0.5000     0.0164   0.0886     0.0924


**Interpretation**

Prior: P(Positive) ≈ 0.500 - the dataset is balanced, so with no keyword the review is a coin flip.

Positive keywords - every posterior sits well above 0.50, confirming positive sentiment:

**keyword P(Positive \| keyword):**

wonderful - 0.812;
excellent - 0.807;
brilliant - 0.761;
masterpiece - 0.730.

Negative keywords: every posterior falls well below 0.50, confirming negative sentiment:

**keyword P(Positive \| keyword):**

boring - 0.194;
awful - 0.098;
worst - 0.092;
waste - 0.069;

Reading the numbers: a review containing 'wonderful' is ~81% likely positive; one containing 'waste' is only ~7% likely positive (i.e. ~93% negative). The positive words land around 0.73–0.81 rather than near 1.0, because they still appear in some negative reviews (negation, sarcasm) - the keyword shifts the odds strongly without guaranteeing the label.